In [1]:
import json
import os
from pathlib import Path
from pprint import pprint
import numpy as np

from datasets import load_from_disk
from transformers import AutoModelForTokenClassification, AutoTokenizer,\
DataCollatorForTokenClassification, TrainingArguments, Trainer, EvalPrediction
import evaluate

import wandb
from colorama import Fore, Style

In [2]:
import torch
torch.cuda.empty_cache()

In [3]:
from transformers.models.deberta_v2.tokenization_deberta_v2 import DebertaV2Tokenizer
from transformers.models.deberta_v2.modeling_deberta_v2 import DebertaV2ForTokenClassification
from datasets import DatasetDict

In [4]:
model_path = "microsoft/deberta-base"
# model_path = "microsoft/deberta-v3-small"
# model_path = "distilbert/distilbert-base-uncased" #temp

In [5]:
DATA_DIR = Path(os.getcwd()).parent / "data"
DATA_DIR.mkdir(exist_ok=True)

LABEL_INFO_PATH = DATA_DIR/"label_info.json"
DATASET_PATH = DATA_DIR/"cleaned_ai4privacy_300k_pii"

MODEL_DIR = Path(os.getcwd()).parent / "models"
MODEL_DIR.mkdir(exist_ok=True)
RUN_NAME = "pii_redaction_" + model_path.replace("/", "_") + "_v1"

In [6]:
dataset: DatasetDict = load_from_disk(DATASET_PATH) # type:ignore
print(f"dataset column names: {dataset["train"].column_names}\n\n")

dataset column names: ['source_text', 'privacy_mask']




In [7]:
# label info
with open(LABEL_INFO_PATH) as f:
    label_info: dict = json.load(f)
label2id = label_info["label2id"]
id2label = label_info["id2label"]

In [8]:
tokenizer: DebertaV2Tokenizer = AutoTokenizer.from_pretrained(model_path)
model: DebertaV2ForTokenClassification = AutoModelForTokenClassification.from_pretrained(
    model_path, num_labels=len(id2label), id2label=id2label, label2id=label2id)

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

[transformers] DebertaForTokenClassification LOAD REPORT from: microsoft/deberta-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [9]:
print(type(model))
print(type(tokenizer))

<class 'transformers.models.deberta.modeling_deberta.DebertaForTokenClassification'>
<class 'transformers.models.deberta.tokenization_deberta.DebertaTokenizer'>


In [10]:
assert tokenizer.is_fast

In [11]:
example = dataset["train"][0]
print("dataset example:")
for item in example.items():
    print(item)

dataset example:
('source_text', 'Subject: Group Messaging for Admissions Process\n\nGood morning, everyone,\n\nI hope this message finds you well. As we continue our admissions processes, I would like to update you on the latest developments and key information. Please find below the timeline for our upcoming meetings:\n\n- wynqvrh053 - Meeting at 10:20am\n- luka.burg - Meeting at 21\n- qahil.wittauer - Meeting at quarter past 13\n- gholamhossein.ruschke - Meeting at 9:47 PM\n- pdmjrsyoz1460 ')
('privacy_mask', [{'value': 'wynqvrh053', 'start': 287, 'end': 297, 'label': 'USERNAME'}, {'value': '10:20am', 'start': 311, 'end': 318, 'label': 'TIME'}, {'value': 'luka.burg', 'start': 321, 'end': 330, 'label': 'USERNAME'}, {'value': '21', 'start': 344, 'end': 346, 'label': 'TIME'}, {'value': 'qahil.wittauer', 'start': 349, 'end': 363, 'label': 'USERNAME'}, {'value': 'quarter past 13', 'start': 377, 'end': 392, 'label': 'TIME'}, {'value': 'gholamhossein.ruschke', 'start': 395, 'end': 416, 'la

In [12]:
ex_tokenized_with_offsets = tokenizer(
    example["source_text"],
    return_offsets_mapping=True,
    add_special_tokens=True,
)
for item in ex_tokenized_with_offsets.items():
    print(item)

('input_ids', [1, 47159, 35, 826, 15212, 6257, 13, 1614, 23363, 19149, 50118, 50118, 12350, 662, 6, 961, 6, 50118, 50118, 100, 1034, 42, 1579, 5684, 47, 157, 4, 287, 52, 535, 84, 18054, 5588, 6, 38, 74, 101, 7, 2935, 47, 15, 5, 665, 5126, 8, 762, 335, 4, 3401, 465, 874, 5, 10589, 13, 84, 2568, 2891, 35, 50118, 50118, 12, 885, 3892, 22638, 20378, 2546, 246, 111, 12776, 23, 158, 35, 844, 424, 50118, 12, 784, 10620, 4, 3321, 111, 12776, 23, 733, 50118, 12, 2231, 895, 718, 4, 605, 2582, 9994, 111, 12776, 23, 297, 375, 508, 50118, 12, 821, 9649, 424, 298, 366, 36353, 4, 14888, 611, 1071, 111, 12776, 23, 361, 35, 3706, 2784, 50118, 12, 181, 43604, 267, 338, 8628, 3979, 1570, 2466, 1437, 2])
('token_type_ids', [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

In [13]:
tokens = ex_tokenized_with_offsets.tokens()
print(tokens)

['[CLS]', 'Subject', ':', 'ĠGroup', 'ĠMess', 'aging', 'Ġfor', 'ĠAd', 'missions', 'ĠProcess', 'Ċ', 'Ċ', 'Good', 'Ġmorning', ',', 'Ġeveryone', ',', 'Ċ', 'Ċ', 'I', 'Ġhope', 'Ġthis', 'Ġmessage', 'Ġfinds', 'Ġyou', 'Ġwell', '.', 'ĠAs', 'Ġwe', 'Ġcontinue', 'Ġour', 'Ġadmissions', 'Ġprocesses', ',', 'ĠI', 'Ġwould', 'Ġlike', 'Ġto', 'Ġupdate', 'Ġyou', 'Ġon', 'Ġthe', 'Ġlatest', 'Ġdevelopments', 'Ġand', 'Ġkey', 'Ġinformation', '.', 'ĠPlease', 'Ġfind', 'Ġbelow', 'Ġthe', 'Ġtimeline', 'Ġfor', 'Ġour', 'Ġupcoming', 'Ġmeetings', ':', 'Ċ', 'Ċ', '-', 'Ġw', 'yn', 'qv', 'rh', '05', '3', 'Ġ-', 'ĠMeeting', 'Ġat', 'Ġ10', ':', '20', 'am', 'Ċ', '-', 'Ġl', 'uka', '.', 'burg', 'Ġ-', 'ĠMeeting', 'Ġat', 'Ġ21', 'Ċ', '-', 'Ġq', 'ah', 'il', '.', 'w', 'itt', 'auer', 'Ġ-', 'ĠMeeting', 'Ġat', 'Ġquarter', 'Ġpast', 'Ġ13', 'Ċ', '-', 'Ġg', 'hol', 'am', 'h', 'os', 'sein', '.', 'rus', 'ch', 'ke', 'Ġ-', 'ĠMeeting', 'Ġat', 'Ġ9', ':', '47', 'ĠPM', 'Ċ', '-', 'Ġp', 'dm', 'j', 'r', 'sy', 'oz', '14', '60', 'Ġ', '[SEP]']


In [14]:
word_ids = ex_tokenized_with_offsets.word_ids()
print(word_ids)

[None, 0, 1, 2, 3, 3, 4, 5, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 58, 58, 58, 59, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 78, 78, 79, 80, 80, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 89, 89, 89, 89, 89, 90, 91, 91, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 101, 101, 101, 101, 101, 102, 102, 103, None]


In [15]:
pprint(example["privacy_mask"], sort_dicts=False)

[{'value': 'wynqvrh053', 'start': 287, 'end': 297, 'label': 'USERNAME'},
 {'value': '10:20am', 'start': 311, 'end': 318, 'label': 'TIME'},
 {'value': 'luka.burg', 'start': 321, 'end': 330, 'label': 'USERNAME'},
 {'value': '21', 'start': 344, 'end': 346, 'label': 'TIME'},
 {'value': 'qahil.wittauer', 'start': 349, 'end': 363, 'label': 'USERNAME'},
 {'value': 'quarter past 13', 'start': 377, 'end': 392, 'label': 'TIME'},
 {'value': 'gholamhossein.ruschke',
  'start': 395,
  'end': 416,
  'label': 'USERNAME'},
 {'value': '9:47 PM', 'start': 430, 'end': 437, 'label': 'TIME'},
 {'value': 'pdmjrsyoz1460', 'start': 440, 'end': 453, 'label': 'USERNAME'}]


In [16]:
def get_ner_labels(batch: dict[str, list]) -> dict[str, list[list[str]]]:
    token_offsets = tokenizer(
        batch["source_text"],
        return_offsets_mapping=True,
        add_special_tokens=True,
        truncation=True
    )["offset_mapping"]

    batch_ner_labels: list[list[str]] = []
    # loop through the batch to get the privacy masks for every sequence
    for i, row_masks in enumerate(batch["privacy_mask"]):
        row_ner_labels = []
        # loop through the token_offsets of the current sequence
        for offset in token_offsets[i]:
            # if add_special_tokens is true skip over special tokens (offset with (0,0))
            if offset == (0, 0): 
                row_ner_labels.append("O")
                continue
            # label is "O" unless privacy label is found
            label = "O" 
            # loop through masks to find label
            for privacy_mask in row_masks:
                # if statement is switched to check if any character of the token falls within the mask
                if offset[1] > privacy_mask["start"] and offset[0] < privacy_mask["end"]:
                    label = privacy_mask["label"]
                    # switch if statement to also include less than
                    if offset[0] <= privacy_mask["start"]:
                        label = "B-" + label
                    else:
                        label = "I-" + label
                    
                    break # break out of for loop when/if label is found
            row_ner_labels.append(label)
            
        batch_ner_labels.append(row_ner_labels)
    
    return {"ner_tags": batch_ner_labels}

In [17]:
dataset = dataset.map(get_ner_labels, batched=True, batch_size=20_000)
pprint(dataset["train"].features)

{'ner_tags': List(Value('string')),
 'privacy_mask': List({'value': Value('string'), 'start': Value('int64'), 'end': Value('int64'), 'label': Value('string')}),
 'source_text': Value('string')}


In [18]:
def tokenize_and_align_labels_single(example: dict[str, list]):
    tokenized_input = tokenizer(
        example["source_text"],
        truncation=True,
        add_special_tokens=True
    )
    labels = []
    word_ids = tokenized_input.word_ids()
    previous_word_idx = None
    for i, tag in enumerate(example["ner_tags"]):
        word_idx = word_ids[i]
        
        if word_idx is None:
            labels.append(-100)
        elif word_idx != previous_word_idx:
            labels.append(label2id[tag])
        else:
            labels.append(-100)
        previous_word_idx = word_idx
            
    tokenized_input["labels"] = labels
    return tokenized_input
            

In [19]:
dataset = dataset.map(tokenize_and_align_labels_single, batched=False)

In [20]:
pprint(dataset["train"].features)

{'attention_mask': List(Value('int8')),
 'input_ids': List(Value('int32')),
 'labels': List(Value('int64')),
 'ner_tags': List(Value('string')),
 'privacy_mask': List({'value': Value('string'), 'start': Value('int64'), 'end': Value('int64'), 'label': Value('string')}),
 'source_text': Value('string'),
 'token_type_ids': List(Value('int8'))}


In [21]:
example = dataset["train"][0]
print(f"{len(example["ner_tags"])=}")
print(f"{len(example["input_ids"])=}")

len(example["ner_tags"])=130
len(example["input_ids"])=130


In [22]:
ex_wi = tokenizer(
    example["source_text"],
    truncation=True,
    add_special_tokens=True
).word_ids()
masks = example["privacy_mask"]

token_offsets = tokenizer(
    example["source_text"],
    return_offsets_mapping=True,
    add_special_tokens=True,
)["offset_mapping"]

id2label = {i: label for label, i in label2id.items()}
tokens = tokenizer.convert_ids_to_tokens(example["input_ids"])

In [23]:
for mask in masks:
    print(Fore.BLUE + f"{mask}" + Style.RESET_ALL)    
for tag, wi, label in zip(example["labels"], ex_wi, token_offsets):
    if tag != -100 and tag != 0:
        token = tokens[wi] #type:ignore
        value = f"label={id2label[tag]}, word_id={wi}, {token=}, {label=}"
        print(value)
        
        masks_lst = []
        for mask in masks:
            if label[1] > mask["start"] and label[0] < mask["end"]:
                masks_lst.append(f"in {mask}")
                
        if masks_lst:
            print(Fore.GREEN + f"{masks_lst}")
            print(Style.RESET_ALL + "", end="")
        else:
            print(Fore.RED + f" {label} not in masks" )
            print(Style.RESET_ALL + "", end="")

{'value': 'wynqvrh053', 'start': 287, 'end': 297, 'label': 'USERNAME'}
{'value': '10:20am', 'start': 311, 'end': 318, 'label': 'TIME'}
{'value': 'luka.burg', 'start': 321, 'end': 330, 'label': 'USERNAME'}
{'value': '21', 'start': 344, 'end': 346, 'label': 'TIME'}
{'value': 'qahil.wittauer', 'start': 349, 'end': 363, 'label': 'USERNAME'}
{'value': 'quarter past 13', 'start': 377, 'end': 392, 'label': 'TIME'}
{'value': 'gholamhossein.ruschke', 'start': 395, 'end': 416, 'label': 'USERNAME'}
{'value': '9:47 PM', 'start': 430, 'end': 437, 'label': 'TIME'}
{'value': 'pdmjrsyoz1460', 'start': 440, 'end': 453, 'label': 'USERNAME'}
label=B-USERNAME, word_id=58, token='Ċ', label=(286, 288)
["in {'value': 'wynqvrh053', 'start': 287, 'end': 297, 'label': 'USERNAME'}"]
label=I-USERNAME, word_id=59, token='Ċ', label=(294, 296)
["in {'value': 'wynqvrh053', 'start': 287, 'end': 297, 'label': 'USERNAME'}"]
label=B-TIME, word_id=63, token='qv', label=(310, 313)
["in {'value': '10:20am', 'start': 311, 'e

In [24]:
example = dataset["train"][0]
example_encodings = tokenizer(
    example["source_text"],
    return_offsets_mapping=True,
    add_special_tokens=True,
)
line1 = ""
line2 = ""
line3 = ""
line4 = ""
for word, tag, word_id, label in zip(example_encodings.tokens(), example["ner_tags"], example_encodings.word_ids(), example_encodings["offset_mapping"]):
    tag = tag
    word_id = str(word_id)
    label = str(label)
    max_length = max(len(word), len(tag), len(word_id), len(label))
    line1 += word + " " * (max_length - len(word) + 1)
    line2 += tag + " " * (max_length - len(tag) + 1)
    line3 += word_id + " " * (max_length - len(word_id) + 1)
    line4 += label + " " * (max_length - len(label) + 1)
print(line1)
print(line2)
print(line3)
print(line4)

[CLS]  Subject :      ĠGroup  ĠMess    aging    Ġfor     ĠAd      missions ĠProcess Ċ        Ċ        Good     Ġmorning ,        Ġeveryone ,        Ċ        Ċ        I        Ġhope    Ġthis    Ġmessage Ġfinds   Ġyou      Ġwell      .          ĠAs        Ġwe        Ġcontinue  Ġour       Ġadmissions Ġprocesses ,          ĠI         Ġwould     Ġlike      Ġto        Ġupdate    Ġyou       Ġon        Ġthe       Ġlatest    Ġdevelopments Ġand       Ġkey       Ġinformation .          ĠPlease    Ġfind      Ġbelow     Ġthe       Ġtimeline  Ġfor       Ġour       Ġupcoming  Ġmeetings  :          Ċ          Ċ          -          Ġw         yn         qv         rh         05         3          Ġ-         ĠMeeting   Ġat        Ġ10        :          20         am         Ċ          -          Ġl         uka        .          burg       Ġ-         ĠMeeting   Ġat        Ġ21        Ċ          -          Ġq         ah         il         .          w          itt        auer       Ġ-         ĠMeeting   Ġat

In [25]:
example = dataset["train"][0]
example_encodings = tokenizer(
    example["source_text"],
    return_offsets_mapping=True,
    add_special_tokens=True,
)
line1 = ""
line2 = ""
line3 = ""
line4 = ""
for word, tag, label, word_id in zip(example_encodings.tokens(), example["ner_tags"], example["labels"], example_encodings.word_ids()):
    word_id = str(word_id)
    label = str(label)
    max_length = max(len(word), len(tag), len(label))
    line1 += word + " " * (max_length - len(word) + 1)
    line2 += tag + " " * (max_length - len(tag) + 1)
    line3 += label + " " * (max_length - len(label) + 1)
    line4 += word_id + " " * (max_length - len(word_id) + 1)
print(line1)
print(line2)
print(line3)
print(line4)

[CLS] Subject : ĠGroup ĠMess aging Ġfor ĠAd missions ĠProcess Ċ Ċ Good Ġmorning , Ġeveryone , Ċ Ċ I Ġhope Ġthis Ġmessage Ġfinds Ġyou Ġwell . ĠAs Ġwe Ġcontinue Ġour Ġadmissions Ġprocesses , ĠI Ġwould Ġlike Ġto Ġupdate Ġyou Ġon Ġthe Ġlatest Ġdevelopments Ġand Ġkey Ġinformation . ĠPlease Ġfind Ġbelow Ġthe Ġtimeline Ġfor Ġour Ġupcoming Ġmeetings : Ċ Ċ - Ġw         yn         qv         rh         05         3          Ġ- ĠMeeting Ġat Ġ10    :      20     am     Ċ - Ġl         uka        .          burg       Ġ- ĠMeeting Ġat Ġ21    Ċ - Ġq         ah         il         .          w          itt        auer       Ġ- ĠMeeting Ġat Ġquarter Ġpast  Ġ13    Ċ - Ġg         hol        am         h          os         sein       .          rus        ch         ke         Ġ- ĠMeeting Ġat Ġ9     :      47     ĠPM    Ċ - Ġp         dm         j          r          sy         oz         14         60         Ġ [SEP] 
O     O       O O      O     O     O    O   O        O        O O O    O        O O     

In [26]:
print([label for label in example["labels"]])

[-100, 0, 0, 0, 0, -100, 0, 0, -100, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, -100, -100, -100, 2, -100, 0, 0, 0, 3, 4, 4, 4, 0, 0, 1, -100, 2, 2, 0, 0, 0, 3, 0, 0, 1, -100, -100, 2, 2, -100, -100, 0, 0, 0, 3, 4, 4, 0, 0, 1, -100, -100, -100, -100, -100, 2, 2, -100, -100, 0, 0, 0, 3, 4, 4, 4, 0, 0, 1, -100, -100, -100, -100, -100, 2, -100, 0, -100]


In [27]:
def validate_label_count(dataset):
    valid = []
    for i in range(len(dataset)):
        labels = dataset[i]["labels"]
        masks = dataset[i]["privacy_mask"]
        b_entities = [label for label in labels if label != -100 or label != 0]
        valid.append(len(b_entities) == len(masks))
    return valid

pprint(validate_label_count(dataset["train"].select(range(5))))

[False, False, False, False, False]


In [28]:
def check_label_distribution(dataset, num_samples=5):
    for i in range(num_samples):
        labels = dataset[i]["labels"]
        valid = [label for label in labels if label != -100]
        print(f"Sample {i}: {len(valid)} valid labels out of {len(labels)} tokens")
        print(f"  Labels: {valid[:10]}")

check_label_distribution(dataset["train"])

Sample 0: 104 valid labels out of 130 tokens
  Labels: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Sample 1: 95 valid labels out of 101 tokens
  Labels: [0, 0, 0, 3, 4, 4, 4, 0, 0, 1]
Sample 2: 100 valid labels out of 121 tokens
  Labels: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Sample 3: 139 valid labels out of 182 tokens
  Labels: [0, 0, 15, 16, 16, 0, 0, 0, 0, 0]
Sample 4: 89 valid labels out of 103 tokens
  Labels: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [29]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

seqeval = evaluate.load("seqeval")

In [30]:
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/zelluzy/.netrc.
wandb: Currently logged in as: bengid (bengid-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [31]:
from collections import Counter
def compute_metrics(p: EvalPrediction) -> dict[str, float]:
    predictions, label_ids = p.predictions, p.label_ids
    predictions = np.argmax(predictions, axis=-1)
    
    true_preds = [
        [id2label[pred] for pred, tgt_id in zip(preds_row, labels_row) if tgt_id != -100]
        for preds_row, labels_row in zip(predictions, label_ids)
    ]
    
    true_labels =[
        [id2label[tgt_id] for tgt_id in labels_row if tgt_id != -100]
        for labels_row in label_ids
    ]
    
    # TODO: remove
    all_preds = [label for seq in true_preds for label in seq]
    print("Prediction distribution:", Counter(all_preds))
    
    results = seqeval.compute(predictions=true_preds, references=true_labels)
    
    flat = {}
    if results is not None:
        flat.update({
            "precision": results["overall_precision"],
            "recall": results["overall_recall"],
            "f1": results["overall_f1"],
            "accuracy": results["overall_accuracy"],
        })
        
        for entity, scores in results.items():
            if isinstance(scores, dict): # filter scores for individual labels
                flat[f"{entity}_f1"] = scores["f1"]
                flat[f"{entity}_support"] = scores["number"]
    
    return flat

In [32]:
# small_train = dataset["train"].select(range(32))

# temp_args = TrainingArguments(
#     output_dir=str(MODEL_DIR),
#     learning_rate=1e-5, # low learning_rate
#     per_device_train_batch_size=16,
#     per_device_eval_batch_size=16,
#     num_train_epochs=1,
#     weight_decay=0.01,
#     eval_strategy="epoch",
#     warmup_ratio=0.1,
#     bf16=False,
#     fp16=False,
#     max_grad_norm=1.0,
# )

# temp_trainer = Trainer(
#     model=model,
#     args=temp_args,
#     train_dataset=small_train,
#     eval_dataset=dataset["validation"],
#     processing_class=tokenizer,
#     data_collator=data_collator,
#     compute_metrics=compute_metrics,
# )
# print(temp_trainer.args.device)
# trainer_output = temp_trainer.train()

In [33]:
num_train_samples = len(dataset["train"])   # 29908
batch_size = 16
num_epochs = 1
total_steps = (num_train_samples // batch_size) * num_epochs  # 1869
warmup = int(0.1 * total_steps)  # 186

training_args = TrainingArguments(
    output_dir=str(MODEL_DIR),
    report_to="wandb",
    run_name=RUN_NAME,
    learning_rate=1e-5, # low learning_rate
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    # metric_for_best_model="eval_f1"
    bf16=False,
    fp16=False,
    max_grad_norm=1.0,
    warmup_steps=warmup
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [34]:
print(trainer.args.device)

cuda:0


In [35]:
wandb.init(project="pii-redaction")
trainer_output = trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy,Bod F1,Bod Support,Building F1,Building Support,City F1,City Support,Country F1,Country Support,Date F1,Date Support,Driverlicense F1,Driverlicense Support,Email F1,Email Support,Geocoord F1,Geocoord Support,Givenname1 F1,Givenname1 Support,Givenname2 F1,Givenname2 Support,Idcard F1,Idcard Support,Ip F1,Ip Support,Lastname1 F1,Lastname1 Support,Lastname2 F1,Lastname2 Support,Lastname3 F1,Lastname3 Support,Pass F1,Pass Support,Passport F1,Passport Support,Postcode F1,Postcode Support,Secaddress F1,Secaddress Support,Sex F1,Sex Support,Socialnumber F1,Socialnumber Support,State F1,State Support,Street F1,Street Support,Tel F1,Tel Support,Time F1,Time Support,Title F1,Title Support,Username F1,Username Support
1,0.044364,0.039884,0.933876,0.941682,0.937763,0.991932,0.960034,1148,0.982350,987,0.980218,1005,0.974725,765,0.916215,838,0.937848,1201,0.986376,1272,0.971429,104,0.768746,954,0.655660,268,0.913990,1352,0.989967,1046,0.770425,1211,0.646075,319,0.617886,106,0.968421,804,0.914639,1213,0.981424,959,0.980660,442,0.973737,979,0.956893,1316,0.986836,1020,0.980493,971,0.957599,1037,0.975610,1876,0.970047,957,0.960212,1331
2,0.036170,0.033140,0.943163,0.956674,0.949870,0.992951,0.963441,1148,0.988889,987,0.978346,1005,0.973394,765,0.927885,838,0.960499,1201,0.989080,1272,0.942857,104,0.846227,954,0.795580,268,0.932061,1352,0.990913,1046,0.824027,1211,0.720379,319,0.723005,106,0.961421,804,0.922957,1213,0.975885,959,0.979545,442,0.973165,979,0.969064,1316,0.985847,1020,0.976507,971,0.967557,1037,0.974386,1876,0.971549,957,0.965879,1331
3,0.035844,0.030326,0.949932,0.962011,0.955933,0.993536,0.969724,1148,0.989930,987,0.980703,1005,0.972418,765,0.944412,838,0.964361,1201,0.984003,1272,0.980769,104,0.886024,954,0.803704,268,0.949614,1352,0.992826,1046,0.826994,1211,0.740399,319,0.685714,106,0.956151,804,0.953573,1213,0.983471,959,0.980660,442,0.974671,979,0.969119,1316,0.987329,1020,0.982060,971,0.968147,1037,0.977027,1876,0.975145,957,0.966942,1331
4,0.012982,0.030075,0.951743,0.962089,0.956888,0.993731,0.967601,1148,0.989410,987,0.982733,1005,0.976864,765,0.944510,838,0.963866,1201,0.980852,1272,0.971154,104,0.873540,954,0.800000,268,0.948507,1352,0.991396,1046,0.843009,1211,0.741573,319,0.746411,106,0.962645,804,0.949139,1213,0.984997,959,0.979592,442,0.974203,979,0.969237,1316,0.989279,1020,0.983590,971,0.971209,1037,0.979365,1876,0.975224,957,0.966242,1331
5,0.030042,0.030914,0.953878,0.961815,0.957830,0.993993,0.967996,1148,0.990909,987,0.985636,1005,0.976083,765,0.944019,838,0.967391,1201,0.987490,1272,0.980769,104,0.875130,954,0.804428,268,0.950407,1352,0.988539,1046,0.845747,1211,0.748844,319,0.732394,106,0.967465,804,0.953431,1213,0.986528,959,0.979638,442,0.973658,979,0.969512,1316,0.988315,1020,0.984078,971,0.969378,1037,0.975623,1876,0.975250,957,0.966917,1331


Prediction distribution: Counter({'O': 329754, 'I-IP': 16233, 'I-EMAIL': 6411, 'I-BOD': 5248, 'I-TEL': 4789, 'I-SOCIALNUMBER': 4192, 'I-TIME': 3952, 'I-PASS': 3647, 'I-DRIVERLICENSE': 3628, 'I-DATE': 3269, 'B-TIME': 1886, 'I-USERNAME': 1694, 'B-LASTNAME1': 1484, 'I-STREET': 1474, 'B-PASSPORT': 1372, 'B-IDCARD': 1342, 'I-IDCARD': 1319, 'B-USERNAME': 1298, 'B-EMAIL': 1288, 'B-SOCIALNUMBER': 1244, 'I-POSTCODE': 1219, 'B-BOD': 1172, 'B-DRIVERLICENSE': 1111, 'B-IP': 1045, 'B-STATE': 1031, 'B-CITY': 1016, 'B-SEX': 1001, 'B-BUILDING': 996, 'B-STREET': 975, 'B-POSTCODE': 972, 'B-TEL': 970, 'B-TITLE': 946, 'I-GEOCOORD': 942, 'B-DATE': 810, 'B-COUNTRY': 778, 'B-PASS': 762, 'B-GIVENNAME1': 630, 'I-CITY': 570, 'I-SECADDRESS': 458, 'B-LASTNAME2': 453, 'B-SECADDRESS': 436, 'I-COUNTRY': 321, 'I-PASSPORT': 286, 'I-SEX': 273, 'B-GIVENNAME2': 153, 'B-LASTNAME3': 140, 'I-LASTNAME1': 113, 'B-GEOCOORD': 99, 'I-GIVENNAME1': 69, 'I-STATE': 52, 'I-GIVENNAME2': 17, 'I-LASTNAME2': 9})


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Prediction distribution: Counter({'O': 329269, 'I-IP': 16240, 'I-EMAIL': 6412, 'I-BOD': 5220, 'I-TEL': 4816, 'I-SOCIALNUMBER': 4175, 'I-TIME': 3993, 'I-PASS': 3763, 'I-DRIVERLICENSE': 3677, 'I-DATE': 3322, 'B-TIME': 1887, 'I-USERNAME': 1722, 'I-STREET': 1489, 'B-PASSPORT': 1339, 'B-USERNAME': 1324, 'B-EMAIL': 1286, 'I-IDCARD': 1274, 'B-SOCIALNUMBER': 1265, 'B-IDCARD': 1257, 'I-POSTCODE': 1242, 'B-DRIVERLICENSE': 1194, 'B-BOD': 1166, 'B-LASTNAME1': 1148, 'B-GIVENNAME1': 1121, 'B-IP': 1045, 'B-STATE': 1029, 'B-CITY': 1022, 'B-SEX': 996, 'B-BUILDING': 993, 'B-STREET': 980, 'B-POSTCODE': 971, 'B-TEL': 971, 'B-TITLE': 941, 'I-GEOCOORD': 941, 'B-DATE': 815, 'B-COUNTRY': 773, 'B-PASS': 767, 'I-CITY': 590, 'I-SECADDRESS': 456, 'B-SECADDRESS': 436, 'I-COUNTRY': 318, 'B-LASTNAME2': 308, 'I-PASSPORT': 293, 'I-SEX': 275, 'B-GIVENNAME2': 274, 'I-GIVENNAME1': 170, 'B-LASTNAME3': 107, 'B-GEOCOORD': 98, 'I-LASTNAME1': 90, 'I-STATE': 52, 'I-GIVENNAME2': 20, 'I-LASTNAME2': 17})


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Prediction distribution: Counter({'O': 329493, 'I-IP': 16245, 'I-EMAIL': 6346, 'I-BOD': 5073, 'I-TEL': 4786, 'I-SOCIALNUMBER': 4182, 'I-TIME': 3992, 'I-PASS': 3759, 'I-DRIVERLICENSE': 3661, 'I-DATE': 3429, 'B-TIME': 1888, 'I-USERNAME': 1751, 'I-STREET': 1486, 'B-IDCARD': 1319, 'B-USERNAME': 1312, 'B-SOCIALNUMBER': 1310, 'I-IDCARD': 1293, 'B-EMAIL': 1276, 'B-PASSPORT': 1272, 'B-LASTNAME1': 1232, 'I-POSTCODE': 1212, 'B-DRIVERLICENSE': 1165, 'B-BOD': 1127, 'B-IP': 1045, 'B-STATE': 1032, 'B-CITY': 1020, 'B-GIVENNAME1': 998, 'B-SEX': 998, 'B-BUILDING': 996, 'B-STREET': 972, 'B-POSTCODE': 971, 'B-TEL': 960, 'B-TITLE': 939, 'I-GEOCOORD': 934, 'B-DATE': 847, 'B-COUNTRY': 789, 'B-PASS': 767, 'I-CITY': 579, 'I-SECADDRESS': 457, 'B-SECADDRESS': 440, 'I-COUNTRY': 332, 'B-LASTNAME2': 299, 'I-PASSPORT': 286, 'B-GIVENNAME2': 277, 'I-SEX': 273, 'I-GIVENNAME1': 135, 'I-LASTNAME1': 104, 'B-LASTNAME3': 103, 'B-GEOCOORD': 98, 'I-STATE': 53, 'I-GIVENNAME2': 18, 'I-LASTNAME2': 16, 'I-TITLE': 2})


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Prediction distribution: Counter({'O': 329492, 'I-IP': 16230, 'I-EMAIL': 6380, 'I-BOD': 5061, 'I-TEL': 4808, 'I-SOCIALNUMBER': 4172, 'I-TIME': 3992, 'I-PASS': 3771, 'I-DRIVERLICENSE': 3689, 'I-DATE': 3457, 'B-TIME': 1875, 'I-USERNAME': 1786, 'I-STREET': 1479, 'B-IDCARD': 1346, 'B-USERNAME': 1313, 'B-SOCIALNUMBER': 1302, 'I-IDCARD': 1291, 'B-EMAIL': 1278, 'B-LASTNAME1': 1243, 'B-PASSPORT': 1226, 'I-POSTCODE': 1200, 'B-DRIVERLICENSE': 1187, 'B-BOD': 1126, 'B-IP': 1042, 'B-STATE': 1034, 'B-CITY': 1012, 'B-SEX': 995, 'B-BUILDING': 993, 'B-POSTCODE': 970, 'B-STREET': 970, 'B-GIVENNAME1': 962, 'B-TEL': 960, 'B-TITLE': 941, 'I-GEOCOORD': 933, 'B-DATE': 852, 'B-COUNTRY': 778, 'B-PASS': 768, 'I-CITY': 579, 'I-SECADDRESS': 458, 'B-SECADDRESS': 442, 'I-COUNTRY': 328, 'B-LASTNAME2': 324, 'B-GIVENNAME2': 274, 'I-SEX': 273, 'I-PASSPORT': 264, 'B-LASTNAME3': 107, 'I-LASTNAME1': 101, 'B-GEOCOORD': 98, 'I-GIVENNAME1': 96, 'I-STATE': 53, 'I-GIVENNAME2': 18, 'I-LASTNAME2': 16, 'I-TITLE': 2, 'I-LASTNAME3'

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [36]:
for i in trainer_output:
    pprint(i)

37385
0.044900008499167955
{'epoch': 5.0,
 'total_flos': 1.995701208580716e+16,
 'train_loss': 0.044900008499167955,
 'train_runtime': 2626.0689,
 'train_samples_per_second': 56.944,
 'train_steps_per_second': 14.236}


In [37]:
out = trainer.predict(dataset["test"]) #type:ignore
preds = np.argmax(out.predictions, axis=-1)
print(preds)
print(Fore.CYAN + "")
pprint(out.metrics)

Prediction distribution: Counter({'O': 329603, 'I-IP': 17114, 'I-EMAIL': 6669, 'I-BOD': 5271, 'I-TEL': 4424, 'I-TIME': 4080, 'I-PASS': 3902, 'I-SOCIALNUMBER': 3823, 'I-DATE': 3804, 'I-DRIVERLICENSE': 3492, 'I-USERNAME': 1977, 'B-TIME': 1894, 'I-POSTCODE': 1581, 'B-USERNAME': 1428, 'I-STREET': 1419, 'B-EMAIL': 1337, 'B-IDCARD': 1301, 'B-PASSPORT': 1266, 'I-IDCARD': 1236, 'B-SOCIALNUMBER': 1218, 'B-LASTNAME1': 1210, 'B-DRIVERLICENSE': 1186, 'B-BOD': 1165, 'B-IP': 1115, 'B-GIVENNAME1': 1114, 'B-SEX': 1040, 'I-GEOCOORD': 1011, 'B-CITY': 1010, 'B-STATE': 990, 'B-TITLE': 983, 'B-BUILDING': 962, 'B-POSTCODE': 955, 'B-STREET': 953, 'B-TEL': 870, 'B-DATE': 853, 'B-COUNTRY': 821, 'B-PASS': 743, 'I-CITY': 573, 'I-SECADDRESS': 434, 'B-SECADDRESS': 416, 'I-COUNTRY': 342, 'I-PASSPORT': 316, 'B-LASTNAME2': 290, 'B-GIVENNAME2': 272, 'I-SEX': 259, 'I-LASTNAME1': 120, 'I-GIVENNAME1': 111, 'B-GEOCOORD': 103, 'B-LASTNAME3': 91, 'I-STATE': 56, 'I-GIVENNAME2': 30, 'I-LASTNAME2': 26, 'I-TITLE': 14, 'I-LASTNA

In [38]:
import torch
for name, param in model.named_parameters():
    if torch.isnan(param).any():
        print(f"NaN in: {name}")
        

In [41]:
import html as html_lib
from typing import Optional


def log_pii_test_results(
    trainer,
    test_dataset,
    tokenizer,
    table_name: str = "pii_test_results",
    run: Optional[wandb.sdk.wandb_run.Run] = None,
) -> wandb.Table:
    id2label = trainer.model.config.id2label
    pred_output = trainer.predict(test_dataset)
    pred_ids = np.argmax(pred_output.predictions, axis=-1)
    true_ids = pred_output.label_ids

    table = wandb.Table(columns=["id", "annotated_sequence", "f1", "precision", "recall", "tp", "fp", "fn"])

    for i in range(len(test_dataset)):
        tokens, true_labels, pred_labels = [], [], []
        for token_id, true_id, pred_id in zip(test_dataset[i]["input_ids"], true_ids[i], pred_ids[i]):
            if true_id == -100:
                continue
            tokens.append(tokenizer.convert_ids_to_tokens(token_id))
            true_labels.append(id2label[str(true_id)])
            pred_labels.append(id2label[str(pred_id)])

        tp = fp = fn = 0
        for true, pred in zip(true_labels, pred_labels):
            if true != "O" and pred == true:  tp += 1
            elif true != "O" and pred != true: fn += 1
            elif true == "O" and pred != "O":  fp += 1

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

        parts = []
        for token, true, pred in zip(tokens, true_labels, pred_labels):
            t = html_lib.escape(token)
            if true != "O" and pred == true:
                badge = f'<sup style="font-size:9px;padding:1px 4px;border-radius:3px;background:#C0DD97;color:#27500A">{html_lib.escape(true)}</sup>'
                parts.append(f'<span style="background:#C0DD97;color:#27500A;font-weight:500;padding:2px 4px;border-radius:3px;margin:1px 2px;display:inline-block">{t}{badge}</span>')
            elif true != "O" and pred != true:
                label = html_lib.escape(f"{true}→{pred}")
                badge = f'<sup style="font-size:9px;padding:1px 4px;border-radius:3px;background:#F7C1C1;color:#791F1F">{label}</sup>'
                parts.append(f'<span style="background:#F7C1C1;color:#791F1F;font-weight:500;padding:2px 4px;border-radius:3px;margin:1px 2px;display:inline-block">{t}{badge}</span>')
            elif true == "O" and pred != "O":
                badge = f'<sup style="font-size:9px;padding:1px 4px;border-radius:3px;background:#FAC775;color:#633806">{html_lib.escape(pred)}</sup>'
                parts.append(f'<span style="background:#FAC775;color:#633806;font-weight:500;padding:2px 4px;border-radius:3px;margin:1px 2px;display:inline-block">{t}{badge}</span>')
            else:
                parts.append(f'<span style="margin:1px 2px;display:inline-block">{t}</span>')

        seq_html = '<div style="font-family:monospace;font-size:12px;line-height:2">' + "".join(parts) + "</div>"
        table.add_data(i, wandb.Html(seq_html), round(f1, 4), round(precision, 4), round(recall, 4), tp, fp, fn)

    (run or wandb).log({table_name: table})
    return table

In [42]:
log_pii_test_results(trainer, dataset["test"], tokenizer)

Prediction distribution: Counter({'O': 329603, 'I-IP': 17114, 'I-EMAIL': 6669, 'I-BOD': 5271, 'I-TEL': 4424, 'I-TIME': 4080, 'I-PASS': 3902, 'I-SOCIALNUMBER': 3823, 'I-DATE': 3804, 'I-DRIVERLICENSE': 3492, 'I-USERNAME': 1977, 'B-TIME': 1894, 'I-POSTCODE': 1581, 'B-USERNAME': 1428, 'I-STREET': 1419, 'B-EMAIL': 1337, 'B-IDCARD': 1301, 'B-PASSPORT': 1266, 'I-IDCARD': 1236, 'B-SOCIALNUMBER': 1218, 'B-LASTNAME1': 1210, 'B-DRIVERLICENSE': 1186, 'B-BOD': 1165, 'B-IP': 1115, 'B-GIVENNAME1': 1114, 'B-SEX': 1040, 'I-GEOCOORD': 1011, 'B-CITY': 1010, 'B-STATE': 990, 'B-TITLE': 983, 'B-BUILDING': 962, 'B-POSTCODE': 955, 'B-STREET': 953, 'B-TEL': 870, 'B-DATE': 853, 'B-COUNTRY': 821, 'B-PASS': 743, 'I-CITY': 573, 'I-SECADDRESS': 434, 'B-SECADDRESS': 416, 'I-COUNTRY': 342, 'I-PASSPORT': 316, 'B-LASTNAME2': 290, 'B-GIVENNAME2': 272, 'I-SEX': 259, 'I-LASTNAME1': 120, 'I-GIVENNAME1': 111, 'B-GEOCOORD': 103, 'B-LASTNAME3': 91, 'I-STATE': 56, 'I-GIVENNAME2': 30, 'I-LASTNAME2': 26, 'I-TITLE': 14, 'I-LASTNA

In [43]:
small_test = dataset["test"].select(range(32))

In [44]:
pred_output = trainer.predict(small_test)
arr = pred_output.predictions

u = np.unique(pred_output.predictions)
print(u)

Prediction distribution: Counter({'O': 2652, 'I-IP': 81, 'I-DATE': 45, 'I-EMAIL': 44, 'I-TIME': 40, 'I-GEOCOORD': 27, 'I-BOD': 25, 'I-SOCIALNUMBER': 20, 'B-TIME': 18, 'I-PASS': 13, 'I-TEL': 12, 'B-LASTNAME1': 11, 'I-USERNAME': 11, 'B-SEX': 11, 'B-DATE': 10, 'B-GIVENNAME1': 10, 'B-TITLE': 8, 'B-USERNAME': 8, 'I-DRIVERLICENSE': 8, 'B-SOCIALNUMBER': 8, 'B-EMAIL': 8, 'B-STATE': 6, 'I-POSTCODE': 6, 'I-STREET': 6, 'B-BOD': 5, 'B-BUILDING': 4, 'B-IDCARD': 4, 'I-GIVENNAME1': 4, 'I-CITY': 4, 'B-IP': 3, 'I-IDCARD': 3, 'B-DRIVERLICENSE': 3, 'B-CITY': 3, 'B-POSTCODE': 3, 'B-SECADDRESS': 3, 'I-SECADDRESS': 3, 'B-GEOCOORD': 3, 'B-COUNTRY': 3, 'B-PASS': 2, 'B-LASTNAME2': 2, 'B-PASSPORT': 2, 'B-STREET': 2, 'I-SEX': 2, 'B-TEL': 2, 'I-LASTNAME1': 2, 'B-GIVENNAME2': 1})
[-100.          -4.031742    -4.0291357 ...   14.9602375   14.967181
   14.979419 ]


In [45]:
wandb.finish()

eval/BOD_f1,▁▃█▆▇
eval/BOD_support,▁▁▁▁▁
eval/BUILDING_f1,▁▆▇▇█
eval/BUILDING_support,▁▁▁▁▁
eval/CITY_f1,▃▁▃▅█
eval/CITY_support,▁▁▁▁▁
eval/COUNTRY_f1,▅▃▁█▇
eval/COUNTRY_support,▁▁▁▁▁
eval/DATE_f1,▁▄███
eval/DATE_support,▁▁▁▁▁
+119,...
